# Getting Started with StarLayer

A short "hello world" tour: install, first parse, first query, first validate. StarLayer is a Python RDF 1.2 wrapper built on rdflib and pyshacl — three packages (`starlayergraph`, `starsparql`, `starshacl`) installed together as one `starlayer` package. This guide is deliberately shallow; each step links to the guide that goes deep on that topic.

## Install

```bash
pip install "git+https://github.com/hidden-graph/starlayer.git"
```

This is the supported public install path. It brings in the graph, SPARQL, and SHACL layers together. If you're working from a local checkout for development instead, see the root [README](../../README.md)'s "Development" section.

In [1]:
from starlayergraph import StarLayerGraph, Namespace
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")

## First parse

`StarLayerGraph` is a drop-in `rdflib.Graph` subclass — `parse()`/`serialize()` work the same way, across ordinary RDF 1.1 Turtle and all eight RDF 1.2 formats. This first example is plain Turtle; the [Graphs guide](02-graphs.ipynb) covers RDF 1.2-specific content (triple terms, reification, direction-tagged literals) and every supported format.

In [2]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ;
      ex:name "Alice" ;
      ex:knows ex:bob .
""", format="turtle")

print("triples:", len(g))
print(g.serialize(format="turtle"))

triples: 3
@prefix ex: <http://example.org/> .

ex:alice a ex:Person ;
    ex:knows ex:bob ;
    ex:name "Alice" .




## First query

`.query()` runs SPARQL directly against the graph. See the [SPARQL guide](03-sparql.ipynb) for RDF-1.2-aware query functions and the query-as-RDF workflow.

In [3]:
rows = g.query("""
    PREFIX ex: <http://example.org/>
    SELECT ?name WHERE { ?person ex:name ?name }
""")
for row in rows:
    print(row.name)

Alice


## First validate

A SHACL shape describes a constraint; `StarShaclValidator().validate()` checks a data graph against it. See the [SHACL shapes guide](04-shacl-shapes.ipynb) for the full processing-mode overview (`validate()`, `apply_rules()`, `evaluate()`, `extract_subgraph()`).

In [4]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=g, shacl_graph=shapes)
print("conforms:", result.conforms)

conforms: True


## Where to go next

- [Graphs](02-graphs.ipynb) — RDF 1.2 semantics, all eight formats, `StarLayerDataset`, canonical hashing.
- [SPARQL](03-sparql.ipynb) — query functions and the query-as-RDF workflow.
- [SHACL shapes](04-shacl-shapes.ipynb) — the four processing modes, and links to node expressions, rules, and subgraph extraction.
- [Working with backend graph databases](06-backend-graph-databases.ipynb) — Oxigraph, Fuseki, and SQL-backed storage instead of the in-memory default used throughout this guide.